In [21]:
import pandas as pd
import numpy as np
from decouple import config
import copy
from pprint import pprint
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns


In [2]:
name_KE = "MUNI"
select_query = "SELECT * FROM elvismunica2024obweekday_export_odbc"
filename='details_project_od_MUNI.xlsx'

In [3]:

def check_home_airport_hotel(df, details_df):

    # Loop through each row in the DataFrame 'df'

    data_list = check_all_characters_present(df,home_airport_hotel_column_names)
    data_list.sort()
    
    for index, row in df.iterrows():
        # Extract latitude and longitude values for origin and destination
        
        origin_addr_lat = row[data_list[6]]
        origin_addr_lng = row[data_list[7]]
        destin_addr_lat = row[data_list[0]]
        destin_addr_lng = row[data_list[1]]

        # Check if origin latitude and longitude are missing
        if not (origin_addr_lat and origin_addr_lng):
            # Get the type of origin place
            origin_place_type = row[data_list[8]]
            lat_col, lng_col = None, None  # Initialize variables for latitude and longitude column names

            # Determine the appropriate columns based on place type
            if 'hotel' in origin_place_type.lower() or 'home' in origin_place_type.lower():
                lat_col, lng_col = data_list[4], data_list[5]
            elif 'airport' in origin_place_type.lower():
                airport_destin_code = row[data_list[2]]
                airport_row = details_df[details_df['LIME_CODE'] == airport_destin_code]
                if not airport_row.empty:
                    lat_col, lng_col = 'lat6', 'lng6'

            # If valid latitude and longitude columns are found, populate the corresponding values
            if lat_col and lng_col:
                df.at[index, data_list[6]] = row[lat_col]
                df.at[index, data_list[7]] = row[lng_col]

        # Check if destination latitude and longitude are missing
        if not (destin_addr_lat and destin_addr_lng):
            # Get the type of destination place
            destin_place_type = row[data_list[3]]
            lat_col, lng_col = None, None  # Initialize variables for latitude and longitude column names

            # Determine the appropriate columns based on place type
            if 'hotel' in destin_place_type.lower() or 'home' in destin_place_type.lower():
                lat_col, lng_col = data_list[4], data_list[5]
            elif 'airport' in destin_place_type.lower():
                airport_destin_code = row[data_list[2]]
                airport_row = details_df[details_df['LIME_CODE'] == airport_destin_code]
                if not airport_row.empty:
                    lat_col, lng_col = 'lat6', 'lng6'

            # If valid latitude and longitude columns are found, populate the corresponding values
            if lat_col and lng_col:
                df.at[index, data_list[0]] = row[lat_col]
                df.at[index, data_list[1]] = row[lng_col]
    # Return the DataFrame 'df' with the populated columns
    return df



In [16]:

def details_dataframe(filename):
    details_df=pd.read_excel(filename,sheet_name='AIR')
    return details_df

home_airport_hotel_column_names=['originaddresslat','originaddresslong', 'destinaddresslat', 'destinaddresslong','originplacetype','homeaddresslat','homeaddresslong',
                                 'destinairportcode','destinplacetype']

columns_check_lat_lng = ['originaddresslat', 'originaddresslong', 'destinaddresslat', 'destinaddresslong']

columns_names=['id', 'originaddresslat', 'originaddresslong', 'prevtran1onbuslat', 'prevtran1onbuslong', 'prevtran1offbuslat', 'prevtran1offbuslong', 'prevtran2onbuslat', 'prevtran2onbuslong', 'prevtran2offbuslat', 'prevtran2offbuslong', 'prevtran3onbuslat', 'prevtran3onbuslong', 'prevtran3offbuslat', 'prevtran3offbuslong', 'prevtran4onbuslat', 'prevtran4onbuslong', 'prevtran4offbuslat', 'prevtran4offbuslong', 'stoponlat', 'stoponlong', 'stopofflat', 'stopofflong', 'nexttran1onbuslat', 'nexttran1onbuslong', 'nexttran1offbuslat', 'nexttran1offbuslong', 'nexttran2onbuslat', 'nexttran2onbuslong', 'nexttran2offbuslat', 'nexttran2offbuslong', 'nexttran3onbuslat', 'nexttran3onbuslong', 'nexttran3offbuslat', 'nexttran3offbuslong', 'nexttran4onbuslat', 'nexttran4onbuslong', 'nexttran4offbuslat', 'nexttran4offbuslong', 'destinaddresslat', 'destinaddresslong']


In [11]:
df=pd.read_csv('elvismunica2024obweekday_export_odbc.csv')

In [17]:
df1=copy.deepcopy(df)
  
details_df=details_dataframe(filename)

df1=check_home_airport_hotel(df1,details_df)

# To delete all Origin and Destin lat lng values if not present
columns_to_check_origin_destin_lat_lng =check_all_characters_present(df1,columns_check_lat_lng)

columns_to_include = check_all_characters_present(df1,columns_names)  # Initialize an empty list




In [25]:
final_results_columns=['id', 'completed', 'intervinit', 'routesurveyedcode', 'routesurveyed', 'have5minforsurvecode', 'have5minforsurve', 'origintransportcode', 'origintransport', 'originplacetype','destintransportcode', 'destintransport', 'destinplacetype', 'elvisstatus', 'elviscomment', 'intrvnote', 'routestatus', 'stopsstatus', 'teststatus']
preprocess_for_final_reviewer_columns=['have5minforsurvecode']

In [23]:

def preprocess_for_Final_Reviewer(df):
    final_reviewer = []
    first_cleaner = []
    final_usage = []
    final_review=[]
    data_list=check_all_characters_present(df,preprocess_for_final_reviewer_columns)
    route_stop_columns=check_all_characters_present(df,['ROUTE_STATUS','Stops_Status'])
    for index, row in df.iterrows():
        interv_init = row['INTERV_INIT']
        have_5_min = row[data_list[0]]

        if route_stop_columns :
            route_status = row['ROUTE_STATUS']
            stops_status = row['Stops_Status']
        else:
            route_status=None
            stops_status=None

        if interv_init == '999':
            first_cleaner.append('Test')
            final_reviewer.append('Test/No 5 MIN')
            final_usage.append('Remove')
            final_review.append(' ')
        elif have_5_min != 1:
            first_cleaner.append('No 5 MIN')
            final_reviewer.append('Test/No 5 MIN')
            final_usage.append('Remove')
            final_review.append(' ')

        elif have_5_min == 1 and route_status == 'Approved' and stops_status == 'Approved':
            final_reviewer.append('HereAPI Approved')
            final_usage.append('Use')
            final_review.append('Approved')
            first_cleaner.append('HereAPI')
        elif have_5_min == 1  and route_status == 'Approved' and stops_status != 'Approved':
            final_reviewer.append(' ')
            final_usage.append(' ')
            final_review.append(' ')
            first_cleaner.append('HereAPI')
        elif have_5_min == 1 and route_status != 'Approved' and stops_status != 'Approved':
            final_reviewer.append(' ')
            final_usage.append(' ')
            final_review.append(' ')
            first_cleaner.append('HereAPI')
        else:
            final_reviewer.append(' ')
            final_usage.append(' ')
            final_review.append(' ')
            first_cleaner.append('')

    new_dict = {'final_reviewer': final_reviewer, 'first_cleaner': first_cleaner, 'final_usage': final_usage,'final_review':final_review}
    return new_dict

In [26]:

final_columns_to_include= check_all_characters_present(df1,final_results_columns)

final_test=df1[final_columns_to_include]

current_date = datetime.now().date()

data=preprocess_for_Final_Reviewer(final_test)

final_test.insert(0, 'Elvis_Date', current_date)
final_test.insert(1,'SUPERVISOR_COMMENT',final_test['ELVIS_COMMENT'])
final_test.insert(1,'route_match_flag','Elvis_Review')
final_test.insert(1,'distance_flag','Elvis_Review')
final_test.insert(1,'REASON FOR REMOVAL [Other]',' ')
final_test.insert(1,'REASON FOR REMOVAL',' ')
final_test.insert(1, 'FINAL_REVIEWER', ' ')
final_test.insert(1, 'Final_Usage', ' ')
final_test.insert(1, '2nd Cleaner', ' ')
final_test.insert(1, '1st Cleaner', ' ')
final_test.insert(1, 'elvis_id', final_test['id'])
final_test.drop_duplicates(subset=['id'],keep='first',inplace=True)

In [31]:
for _,row in final_test.iterrows():
    interv_init = row['INTERV_INIT']
    have_5_min = row['HAVE_5_MIN_FOR_SURVECode']
    if interv_init=='999':
        final_test.loc[row.name,'1st Cleaner']='Test'        
        final_test.loc[row.name,'FINAL_REVIEWER']='Test/No 5 MIN'
    elif have_5_min!=1:
        final_test.loc[row.name,'1st Cleaner']='No 5 MIN'        
        final_test.loc[row.name,'FINAL_REVIEWER']='Test/No 5 MIN'
    else:
        final_test.loc[row.name,'1st Cleaner']='HereAPI'        
        final_test.loc[row.name,'FINAL_REVIEWER']=''

In [30]:
final_test.head()

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,Final_Usage,FINAL_REVIEWER,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],distance_flag,route_match_flag,...,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,ELVIS_STATUS,ELVIS_COMMENT
0,2024-06-28,5844,,,,,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-06-28,5851,,,,,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-06-28,5854,,,,,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-06-28,5858,,,,,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-06-28,5864,,,,,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
final_test.head(25)

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,Final_Usage,FINAL_REVIEWER,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],distance_flag,route_match_flag,...,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,ELVIS_STATUS,ELVIS_COMMENT
0,2024-06-28,5844,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-06-28,5851,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-06-28,5854,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-06-28,5858,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-06-28,5864,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-06-28,5866,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-06-28,5875,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-06-28,5884,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-06-28,5916,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2024-06-28,5928,Test,,,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
old_ke=pd.read_excel('MUNI_CA_KINGElvis (3).xlsx',sheet_name='Elvis_Review')
old_ke.head(2)

,elvis_date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,FINAL_REVIEWER2,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,...,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO
0,2024-03-01,5844.0,Test/No 5 MIN,NaN,Remove,NaN,Test/No 5 MIN,NaN,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,2024-03-01,5851.0,Test/No 5 MIN,NaN,Remove,NaN,Test/No 5 MIN,NaN,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [79]:
new_ke=pd.read_csv('MUNI_KINGElvis_no_aa.csv')

In [80]:
new_ke.head(2)

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,Final_Usage,FINAL_REVIEWER,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],distance_flag,route_match_flag,...,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status
0,2024-06-28,5844,Test,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested
1,2024-06-28,5851,Test,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested


In [81]:
new_df=pd.merge(new_ke, old_ke, on='elvis_id', how='left', indicator=True)

In [82]:
filtered_df = new_df[new_df['_merge'] == 'left_only']

In [83]:
filtered_df = filtered_df.drop(columns=['_merge'])

In [84]:
filtered_df

,Elvis_Date,elvis_id,1st Cleaner_x,2nd Cleaner,Final_Usage_x,FINAL_REVIEWER_x,REASON FOR REMOVAL_x,REASON FOR REMOVAL [Other]_x,distance_flag_x,route_match_flag_x,...,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO
3873,2024-06-28,16638,No 5 MIN,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3875,2024-06-28,16640,No 5 MIN,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3885,2024-06-28,16657,No 5 MIN,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3886,2024-06-28,16658,No 5 MIN,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3887,2024-06-28,16659,No 5 MIN,,Remove,Test/No 5 MIN,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26628,2024-06-28,1000477,HereAPI,,,,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26629,2024-06-28,1000478,HereAPI,,,,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26630,2024-06-28,1000479,HereAPI,,,,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26631,2024-06-28,1000480,HereAPI,,,,,,Elvis_Review,Elvis_Review,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
filtered_df.to_csv("MUNI_Missing_New_Records.csv",index=False)